# PLS_Regression — otimização walk-forward

A seleção usa apenas os 80% de desenvolvimento. Em cada origem de validação o modelo é reajustado e prevê somente a próxima hora.

**Esta etapa é pesada e possui checkpoint em CSV.** Se interromper, rode novamente para continuar dos candidatos já concluídos.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
ranking_path = config.RESULT_DIR / "tuning_candidates_PLS_Regression.csv"

winner, ranking = models.tune_pls(model_df, ranking_path)
save_json(winner, config.RESULT_DIR / "best_params_PLS_Regression.json")
all_params_path = config.RESULT_DIR / "best_params.json"
all_params = load_json(all_params_path) if all_params_path.exists() else {}
all_params["PLS_Regression"] = winner
save_json(all_params, all_params_path)
print("Vencedor:", winner)
display(ranking)



PLS_Regression candidato 1/93: {'feature_set': 'full', 'n_components': 1}
PLS_Regression: 20/258 origens | MAE parcial=506.298
PLS_Regression: 40/258 origens | MAE parcial=424.036
PLS_Regression: 60/258 origens | MAE parcial=410.241
PLS_Regression: 80/258 origens | MAE parcial=458.797
PLS_Regression: 100/258 origens | MAE parcial=430.672
PLS_Regression: 120/258 origens | MAE parcial=468.810
PLS_Regression: 140/258 origens | MAE parcial=457.317
PLS_Regression: 160/258 origens | MAE parcial=442.546
PLS_Regression: 180/258 origens | MAE parcial=442.113
PLS_Regression: 200/258 origens | MAE parcial=429.074
PLS_Regression: 220/258 origens | MAE parcial=430.707
PLS_Regression: 240/258 origens | MAE parcial=434.956
PLS_Regression: 258/258 origens | MAE parcial=436.807

PLS_Regression candidato 2/93: {'feature_set': 'full', 'n_components': 2}
PLS_Regression: 20/258 origens | MAE parcial=346.822
PLS_Regression: 40/258 origens | MAE parcial=326.117
PLS_Regression: 60/258 origens | MAE parcial=2

,model,parameters,validation_mae,mean_seconds_per_origin,elapsed_seconds,convergence_rate,status,protocol_signature,feature_complexity,component_complexity,mae_3
0,PLS_Regression,"{""feature_set"": ""no_weather"", ""n_components"": 15}",233.262602,0.015966,4.119187,1.0,ok,96a32d05b15356f0,32,15,233.263
1,PLS_Regression,"{""feature_set"": ""full"", ""n_components"": 30}",233.416621,0.026905,6.941486,1.0,ok,96a32d05b15356f0,36,30,233.417
2,PLS_Regression,"{""feature_set"": ""full"", ""n_components"": 31}",233.431671,0.027146,7.003726,1.0,ok,96a32d05b15356f0,36,31,233.432
3,PLS_Regression,"{""feature_set"": ""full"", ""n_components"": 32}",233.474441,0.029062,7.498064,1.0,ok,96a32d05b15356f0,36,32,233.474
4,PLS_Regression,"{""feature_set"": ""full"", ""n_components"": 29}",233.483304,0.025871,6.674697,1.0,ok,96a32d05b15356f0,36,29,233.483
...,...,...,...,...,...,...,...,...,...,...,...
88,PLS_Regression,"{""feature_set"": ""full"", ""n_components"": 2}",381.609921,0.008992,2.319916,1.0,ok,96a32d05b15356f0,36,2,381.610
89,PLS_Regression,"{""feature_set"": ""compact"", ""n_components"": 2}",382.436860,0.007301,1.883716,1.0,ok,96a32d05b15356f0,29,2,382.437
90,PLS_Regression,"{""feature_set"": ""compact"", ""n_components"": 1}",427.711449,0.006953,1.793886,1.0,ok,96a32d05b15356f0,29,1,427.711
91,PLS_Regression,"{""feature_set"": ""full"", ""n_components"": 1}",436.806700,0.013652,3.522100,1.0,ok,96a32d05b15356f0,36,1,436.807


O teste final não foi usado nesta escolha. Revise o ranking antes de executar o notebook de walk-forward final.